# Fold Change Analysis of Metabolomics Data with PySirius

This notebook demonstrates how to perform a fold change analysis of LC-MS/MS metabolomics data using the [PySirius](https://github.com/sirius-ms/sirius-client-openAPI/tree/fold-change-test) Python client for the [SIRIUS](https://bio.informatik.uni-jena.de/software/sirius/) platform.

We use the publicly available rosemary (*Salvia rosmarinus*) dataset from MassIVE ([MSV000080553](https://massive.ucsd.edu/ProteoSAFe/dataset.jsp?task=9a75f69f0f33460293ee3928361ab836)), originally published by Protsyuk *et al.* (2018, *Nature Protocols*). As a proof-of-concept, we replicate the reported finding that rosmarinic acid is more abundant in old leaves than in young leaves.

The analysis proceeds in two stages:

1. **Feature-level analysis**: locating a specific annotated compound (rosmarinic acid) and quantifying its fold change between tissue age groups.
2. **Compound class-level analysis**: obtaining a dataset-wide overview of which chemical classes are differentially abundant between young and old leaf tissue, using both NPC and ClassyFire classifications.

---

**Reference dataset:** https://massive.ucsd.edu/ProteoSAFe/dataset.jsp?task=9a75f69f0f33460293ee3928361ab836  
**Reference paper:** Protsyuk *et al.* (2018) *Nature Protocols* 13, 134-170. https://doi.org/10.1038/nprot.2017.122  
**PySirius repository:** https://github.com/sirius-ms/sirius-client-openAPI/tree/star-protocol

## 1. Setup

### Installation of PySirius

Install the `star-protocol` branch directly from GitHub via pip:

```shell
pip install "git+https://github.com/sirius-ms/sirius-client-openAPI@star-protocol#subdirectory=client-api_python/generated"
```

Note that this variant will not include SIRIUS.

### Download of the dataset from MassIVE

The dataset can be downloaded from the MassIVE FTP server using Python's built-in `ftplib`. The function below recursively mirrors the remote directory, preserving the folder structure locally. You may also download the dataset any other way.

**Note:** The rest of this notebook assumes the **data to be saved under ./MSV000080553**. If it is not, please adjust the paths accordingly. The path will be correct when using the below python function for download.

In [ ]:
# import ftplib
# import os
#
# def download_ftp_dir(ftp, remote_dir, local_dir):
#     os.makedirs(local_dir, exist_ok=True)
#     lines = []
#     ftp.retrlines(f"LIST {remote_dir}", lines.append)
#     for line in lines:
#         parts = line.split(maxsplit=8)
#         kind = line[0]
#         name = parts[8].split(" -> ")[0]
#         remote_path = f"{remote_dir}/{name}"
#         local_path = os.path.join(local_dir, name)
#         if kind in ("d", "l"):
#             download_ftp_dir(ftp, remote_path, local_path)
#         else:
#             with open(local_path, "wb") as f:
#                 ftp.retrbinary(f"RETR {remote_path}", f.write)
#
# ftp = ftplib.FTP("massive-ftp.ucsd.edu")
# ftp.login()
# download_ftp_dir(ftp, "/v01/MSV000080553", "./MSV000080553")
# ftp.quit()

### Connecting to SIRIUS

SIRIUS does not have to be running before executing this notebook. `SiriusSDK` will attempt to locate and start it automatically via the `SIRIUS_EXE` environment variable or the system path. If SIRIUS is not in your path, use `sdk.start_sirius(sirius_path="...", port=...)`. To attach to an already-running instance, use `sdk.attach_to_sirius(sirius_major_version=6, port=...)`.

If you want to use a specific SIRIUS version, you can start your SIRIUS in REST mode via `sirius rest -s --headless`. You might also specify a port using the `-p PORT_NUMBER` parameter, where wou choose a port number that is not already in use for `PORT_NUMBER`, like `sirius rest -s -p 8080 --headless`

In [ ]:
from PySirius import *

In [ ]:
sdk = SiriusSDK()
api = sdk.attach_or_start_sirius()

if api.actuator().health().get('status') != "UP":
    raise RuntimeError("SIRIUS REST service is not reachable. Check that SIRIUS started correctly.")

### Login to SIRIUS

SIRIUS is free for academic use. Register or log in with your institutional email address. Ensure that you genuinely accept the terms of usage before running the cell below.

Running `sign_up` will respond with the link to the sign-up webpage as well as attempt to open the browser window on that webpage.

In [ ]:
# api.account().sign_up()

In [ ]:
# accept_terms = True
# account_credentials = AccountCredentials(username='YOUR_EMAIL', password='YOUR_PASSWORD')
# api.account().login(accept_terms, account_credentials)

## 2. Project and Data

### 2.1 Creating a New Project Space

A SIRIUS project space is a `.sirius` file that stores all input data and annotation results. The cell below creates a new, empty project space.

In [ ]:
project_info = api.projects().create_project("STAR_protocol_rosemary")

### 2.2 Opening an Existing Project Space

If you already have the pre-processed rosemary dataset available as a `.sirius` project, you can open it directly and skip Sections 2.3 and 2.4. Uncomment the cell below and adjust the path accordingly. `open_project` is used here because the project already exists on disk; `create_project` is only needed when starting from scratch.

In [ ]:
# project_info = api.projects().open_project(
#     "STAR_rosemary",
#     "path/to/project"
# )

### 2.3 Enumerate Run Files

Collect all mzXML and mzML files from the peak-picked data folder. The raw mzXML files are located under the MassIVE FTP mirror ([MSV000080553](https://massive.ucsd.edu/ProteoSAFe/dataset.jsp?task=9a75f69f0f33460293ee3928361ab836)), organised into sample groups by plant part (stem, leaf, flower) and developmental age (zones Z1-Z10).

Set `root_path` to the local MassIVE dataset directory. If you used the download cell above, the default value is correct.

In [ ]:
root_path = "./MSV000080553"

In [ ]:
from os import listdir
from os.path import isfile, join

folder_path = f"{root_path}/peak/Rosumarinus"
files = [
    join(folder_path, f)
    for f in listdir(folder_path)
    if isfile(join(folder_path, f)) and f.endswith((".mzXML", ".mzML"))
]
print(f"{len(files)} run files found")

### 2.4 Parse Sample Group Metadata

The dataset ships with a flat key=value metadata file that maps group names to lists of filenames. We parse this file and perform three sanity checks:

1. Blanks and samples do not overlap.
2. Flower, stem, and leaf groups are mutually exclusive.
3. Young and old groups do not overlap.

**Note:** The current MassIVE archive includes a corrected mapping under `/updates/2026-03-25_lfnothias_1828a86c/metadata/group_mapping_rosemary_corrected.txt`. We intentionally use the **original** mapping file that **includes an overlap** in the grouping here for educational purpose.

In [ ]:
def parse_group_mapping(file_path):
    groups = {}
    with open(file_path) as f:
        for line in f:
            line = line.strip()
            if not line or '=' not in line:
                continue
            key, value = line.split('=', 1)
            groups[key] = value.split(';')
    return groups

groups = parse_group_mapping(f"{root_path}/other/group_mapping_rosemary.txt")
print("Groups found:", list(groups.keys()))

In [ ]:
def are_groups_disjoint(*group_keys, groups):
    intersection = set(groups[group_keys[0]])
    for key in group_keys[1:]:
        intersection &= set(groups[key])
    return len(intersection) == 0

def report_group_intersections(*group_keys, groups):
    value_to_groups = {}
    for key in group_keys:
        for value in groups[key]:
            value_to_groups.setdefault(value, []).append(key)
    return {v: g for v, g in value_to_groups.items() if len(g) > 1}

print("Blanks ∩ Samples = ∅:",       are_groups_disjoint('GROUP_BLANKS', 'GROUP_SAMPLES', groups=groups))
print("Flowers ∩ Stem ∩ Leaf = ∅:",  are_groups_disjoint('GROUP_FLOWERS', 'GROUP_STEM', 'GROUP_LEAF', groups=groups))
print("Young ∩ Old = ∅:",            are_groups_disjoint('GROUP_YOUNG', 'GROUP_OLD', groups=groups))

Check 3 reveals a bug in the original metadata file: the intersection between `GROUP_YOUNG` and `GROUP_OLD` is not empty, meaning some files are assigned to both classes simultaneously. This is logically impossible and must be corrected before proceeding.

In [ ]:
import re

z_groups = [f'GROUP_Z{i+1}' for i in range(10)]
print("Intersecting Z-groups:", report_group_intersections(*z_groups, groups=groups))

def extract_z_values(filenames):
    return [f"Z{m.group(1)}" for f in filenames if (m := re.search(r'_Z(\d+)_', f))]

print("Z-values in GROUP_YOUNG:", extract_z_values(groups['GROUP_YOUNG']))
print("Z-values in GROUP_OLD:",   extract_z_values(groups['GROUP_OLD']))

The output confirms a substring collision: two Z10 files appear in both `GROUP_Z1` (and therefore `GROUP_YOUNG`) and `GROUP_Z10` / `GROUP_OLD`. The Z-value printout also verifies that low Z-numbers (Z1-Z5) correspond to young tissue and high Z-numbers (Z6-Z10) to old tissue, consistent with the original publication. Both misassigned files are removed from `GROUP_Z1` and `GROUP_YOUNG`.

In [ ]:
intersecting = report_group_intersections('GROUP_YOUNG', 'GROUP_OLD', groups=groups)
for filename in intersecting:
    groups['GROUP_Z1'].remove(filename)
    groups['GROUP_YOUNG'].remove(filename)

print("Young ∩ Old = ∅ after correction:", are_groups_disjoint('GROUP_YOUNG', 'GROUP_OLD', groups=groups))

### 2.5 Import Data into SIRIUS

All run files are imported into the project space using `import_ms_run_data_as_job`, which accepts a list of mzXML/mzML files together with a `LcmsSubmissionParameters` object. Classifying runs as `"Sample"` or `"Blank"` allows SIRIUS to subtract blank signal during LC-MS feature alignment. The import runs as an asynchronous job; `wait_for_job_completion` blocks until alignment is finished.

**Note:** The sample types are mapped positionally to the file list; both lists must remain in the same order for blank subtraction to work correctly.

In [ ]:
from os.path import basename

sample_types = [
    "Blank" if basename(f) in groups['GROUP_BLANKS'] else "Sample"
    for f in files
]

submission_parameters = LcmsSubmissionParameters.from_dict({"sampleTypes": sample_types})
import_job = api.projects().import_ms_run_data_as_job(project_info.project_id, files, submission_parameters)
api.wait_for_job_completion(project_info.project_id, import_job.id)

## 3. Data Quality and Computation

SIRIUS assigns each aligned feature a `DataQuality` flag. `DataQuality` is an enum class; iterating over it reveals all available tiers.

The hierarchy is GOOD > DECENT > BAD > NOT_APPLICABLE > LOWEST, where NOT_APPLICABLE might for example be a feature missing its MS2.

In [ ]:
aligned_features = api.features().get_aligned_features(
    project_info.project_id
)

In [ ]:
[q.value for q in DataQuality]

The cell below plots the quality distribution across all features before any exclusion. Inspect this distribution to inform your choice of quality threshold in the next step.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

QUALITY_ORDER = [q.value for q in DataQuality]

palette = px.colors.qualitative.Plotly

QUALITY_COLORS = {
    quality: palette[i % len(palette)]
    for i, quality in enumerate(QUALITY_ORDER)
}

quality_counts = Counter(f.quality for f in aligned_features)

fig = go.Figure(go.Bar(
    x=QUALITY_ORDER,
    y=[quality_counts.get(k, 0) for k in QUALITY_ORDER],
    marker_color=[QUALITY_COLORS.get(k) for k in QUALITY_ORDER],
    hovertemplate="%{x}: %{y} features<extra></extra>"
))

fig.update_layout(
    title="Feature Quality Distribution (all features, before exclusion)",
    xaxis_title="Data Quality",
    yaxis_title="Number of Aligned Features",
    showlegend=False,
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor="white"
)
fig.show()

### 3.2 Quality-Based Feature Exclusion

Based on the distribution above, we now exclude features at or below a chosen quality tier. Set `EXCLUDED_QUALITIES` to any list of `DataQuality` values; all listed tiers will be removed.

> **This is a critical step that shapes every downstream result in this notebook.** The feature IDs of the surviving features are passed directly to the SIRIUS computation job in Section 3.3 via `job_submission.aligned_feature_ids`. This means that formula prediction, CSI:FingerID structure annotation, and CANOPUS compound class prediction are run exclusively on the quality-filtered feature set. Consequently, all fold change computations, compound annotations, and class-level statistics produced in Sections 4 and 5 are based entirely on this filtered set. Features excluded here will not appear in any result, table, or plot.

In [ ]:
# List every DataQuality tier to be excluded from all downstream analysis.
EXCLUDED_QUALITIES = [DataQuality.NOT_APPLICABLE, DataQuality.LOWEST]

if EXCLUDED_QUALITIES:
    aligned_features = [
        f for f in aligned_features
        if f.quality not in EXCLUDED_QUALITIES
    ]
    print(f"Excluded tiers: {[q.value for q in EXCLUDED_QUALITIES]}")
else:
    print("No quality exclusion applied.")

print(f"{len(aligned_features)} features remaining after quality filtering")

### 3.3 Run SIRIUS Computations

We retrieve the default job configuration as a `JobSubmission` object, which enables formula prediction (SIRIUS), structure annotation (CSI:FingerID), and compound class prediction (CANOPUS) by default. Individual tools can be toggled before submission.

Rather than writing the configuration to a file, we save it directly into SIRIUS under a custom name using `save_job_config`. This makes the configuration a first-class citizen in SIRIUS: it persists in the running instance, can be inspected in the GUI, and can be passed by name to `start_job_from_config` to start a run without re-submitting the full object. This is the recommended workflow for reproducible or repeated analyses.

**`job_submission.aligned_feature_ids` is set to the IDs of the quality-filtered features from Section 3.2. This restricts all SIRIUS computations to exactly this feature set. All fold change results, compound annotations, and class-level statistics produced in subsequent sections are therefore grounded in the same quality-filtered data.**

In [ ]:
job_submission = api.jobs().get_default_job_config(include_config_map=True)
job_submission

In [ ]:
JOB_CONFIG_NAME = "rosemary_fold_change_analysis"

# Example: change the instrument profile
# job_submission.formula_id_params.profile = InstrumentProfile.ORBITRAP

# Example: change MS2 mass accuracy
# job_submission.formula_id_params.mass_accuracy_ms2ppm = 5

# Example: enable de-novo structure prediction
# job_submission.ms_novelist_params.enabled = True

# Example: restrict structure database search
# job_submission.structure_db_search_params.structure_search_dbs = ['CHEBI', 'COCONUT']

api.jobs().save_job_config(JOB_CONFIG_NAME, job_submission)
job = api.jobs().start_job_from_config(project_info.project_id, JOB_CONFIG_NAME, [f.aligned_feature_id for f in aligned_features])
api.wait_for_job_completion(project_info.project_id, job.id)

## 4. Sample Tagging and Group Definition

SIRIUS uses a tag-based system to annotate runs with arbitrary metadata and to define sample groups for statistical comparisons. Tags are typed key-value pairs attached to runs. Here we define two categorical dimensions: `plantPart` (Flower / Leaf / Stem) and `age` (Young / Old). Once tags are applied, named groups are created using Lucene query syntax over the tag space, enabling flexible Boolean combinations (e.g., `tags.age:"Young" AND tags.plantPart:"Leaf"`). These group names are subsequently passed directly to fold change submissions.

In [ ]:
def create_run_id_groups(runs, groups):
    name_to_id = {run.name: run.run_id for run in runs.content}
    return {
        group_name: [
            name_to_id[f.rsplit('.', 1)[0]]
            for f in filenames
            if f.rsplit('.', 1)[0] in name_to_id
        ]
        for group_name, filenames in groups.items()
    }

runs = api.runs().get_runs_page_experimental(project_info.project_id, size=len(files))
id_groups = create_run_id_groups(runs, groups)

In [ ]:
plant_part_tag = TagDefinitionImport.from_dict({
    "tagName": "plantPart",
    "tagType": "PROCESSING",
    "valueType": ValueType.TEXT,
    "possibleValues": ["Flower", "Leaf", "Stem"]
})

age_tag = TagDefinitionImport.from_dict({
    "tagName": "age",
    "tagType": "PROCESSING",
    "valueType": ValueType.TEXT,
    "possibleValues": ["Young", "Old"]
})

api.tags().create_tags(project_info.project_id, [plant_part_tag, age_tag])

In [ ]:
tag_map = [
    ('GROUP_STEM',    'plantPart', 'Stem'),
    ('GROUP_FLOWERS', 'plantPart', 'Flower'),
    ('GROUP_LEAF',    'plantPart', 'Leaf'),
    ('GROUP_YOUNG',   'age',       'Young'),
    ('GROUP_OLD',     'age',       'Old'),
]

tag_submissions = [
    TagSubmission.from_dict({"tagName": tag_name, "value": value, "taggedObjectId": run_id})
    for group_key, tag_name, value in tag_map
    for run_id in id_groups[group_key]
]

api.runs().add_tags_to_runs_experimental(project_info.project_id, tag_submissions)

In [ ]:
api.tags().add_group(project_info.project_id, "Young Leaves", 'tags.age:"Young" AND tags.plantPart:"Leaf"', "Samples")
api.tags().add_group(project_info.project_id, "Old Leaves",   'tags.age:"Old" AND tags.plantPart:"Leaf"',   "Samples")

## 5. Feature-Level Fold Change Analysis

Fold changes are computed at the aligned feature level, comparing *Young Leaves* versus *Old Leaves*. This identifies individual metabolic features that are differentially abundant between the two tissue age groups.

When submitting a fold change job, three aggregation strategies (average, maximum, minimum) and two quantification measures (area under the curve, apex intensity) can be specified. All requested combinations are computed in a single job and retrieved independently by specifying the desired combination at retrieval time. Without explicit specification, the default is average area under the curve.

Each fold change entry also includes `leftAbundances` and `rightAbundances`: the aggregated, quantified values for the left and right groups respectively. These are the values directly used to compute the fold change ratio.

In [ ]:
[a.value for a in AggregationType]

In [ ]:
[q.value for q in QuantMeasure]

In [ ]:
fold_change_submission = FoldChangeJobSubmission.from_dict({
    "leftRunGroup": "Young Leaves",
    "rightRunGroup": "Old Leaves",
    "aggregationTypes": [AggregationType.AVG],
    "quantificationMeasures": [QuantMeasure.AREA_UNDER_CURVE]
})

fc_job = api.feature_statistics().compute_aligned_feature_fold_changes_experimental(
    project_info.project_id, fold_change_submission)
api.wait_for_job_completion(project_info.project_id, fc_job.id)

**Note:** There are multiple aggregation types and quantification measures to choose from. The commented-out code below shows how to request all combinations in a single job submission. Choose parameters according to your analysis needs.

In [ ]:
# fold_change_submission = FoldChangeJobSubmission.from_dict({
#     "leftRunGroup": "Young Leaves",
#     "rightRunGroup": "Old Leaves",
#     "aggregationTypes": [AggregationType.AVG, AggregationType.MAX, AggregationType.MIN],
#     "quantificationMeasures": [QuantMeasure.AREA_UNDER_CURVE, QuantMeasure.APEX_INTENSITY]
# })

In [ ]:
feature_statistics_table = api.feature_statistics().get_aligned_feature_fold_change_table_experimental(
    project_info.project_id,
    AggregationType.AVG,
    QuantMeasure.AREA_UNDER_CURVE
)
feature_statistics_table

### 5.1 Replicating the Rosmarinic Acid Finding

Protsyuk *et al.* (2018) reported that rosmarinic acid (PubChem CID 5281792) is more abundant in old leaves of rosemary. We locate this compound among the SIRIUS structure annotations and retrieve its fold change.

The PubChem CID for rosmarinic acid (5281792) can be looked up on the [PubChem website](https://pubchem.ncbi.nlm.nih.gov/compound/5281792). To keep the workflow self-contained, we resolve it programmatically via `pubchempy`.

In [ ]:
import pubchempy as pcp

results = pcp.get_compounds('Rosmarinic acid', 'name')
if not results:
    raise RuntimeError("Rosmarinic acid not found in PubChem.")
rosmarinic_acid_pubchem_id = results[0].cid
print(f"PubChem CID for rosmarinic acid: {rosmarinic_acid_pubchem_id}")

In [ ]:
aligned_features = api.features().get_aligned_features(
    project_info.project_id,
    opt_fields=[AlignedFeatureOptField.TOPANNOTATIONS]
)

In [ ]:
pubchem_db_link = DBLink.from_dict({'name': 'PUBCHEM', 'id': str(rosmarinic_acid_pubchem_id)})

rosmarinic_acid_features = [
    f for f in aligned_features
    if f.top_annotations.structure_annotation is not None
    and pubchem_db_link in f.top_annotations.structure_annotation.db_links
]

print(f"Found {len(rosmarinic_acid_features)} features annotated as rosmarinic acid:")
for f in rosmarinic_acid_features:
    adduct = f.top_annotations.formula_annotation.adduct
    print(f"  {f.aligned_feature_id}  {adduct}")

Multiple hits are expected when the same compound is detected as different adduct ions (e.g., [M+H]+, [M+Na]+, [M-H]-). We select the protonated molecule [M+H]+ as the primary feature, as it is the most commonly observed adduct in positive-mode ESI and serves as the reference for quantification.

In [ ]:
rosmarinic_acid_base_feature = next(
    f for f in rosmarinic_acid_features
    if f.top_annotations.formula_annotation.adduct == '[M + H]+'
)
print(f"Base feature ([M+H]+): {rosmarinic_acid_base_feature.aligned_feature_id}")
print(f"Base feature quality:  {rosmarinic_acid_base_feature.quality}")

In [ ]:
import math

def get_fold_change(statistics_table, row_id, group1, group2):
    """Return the log2 fold change of group1 relative to group2 for a given feature."""
    try:
        row_idx = statistics_table.row_ids.index(row_id)
    except ValueError:
        return None
    for col_idx, (left, right) in enumerate(zip(
        statistics_table.column_left_groups,
        statistics_table.column_right_groups
    )):
        if left == group1 and right == group2:
            fc = statistics_table.values[row_idx][col_idx]
            return math.log2(fc) if fc > 0 else None
        elif left == group2 and right == group1:
            fc = statistics_table.values[row_idx][col_idx]
            return -math.log2(fc) if fc > 0 else None
    return None


log2_fc = get_fold_change(
    feature_statistics_table,
    rosmarinic_acid_base_feature.aligned_feature_id,
    "Old Leaves",
    "Young Leaves"
)
print(f"Rosmarinic acid log2(Old / Young) = {log2_fc:.3f}  (fold change = {2**log2_fc:.2f}x)")

As we reverted the order from Young/Old to Old/Young here, a positive log2 fold change confirms that rosmarinic acid is more abundant in old leaf tissue, replicating the finding of Protsyuk *et al.* (2018).

### 5.2 Highest Differentially Abundant Features

Beyond the targeted rosmarinic acid query, we rank all features that received a structural annotation from CSI:FingerID by the **absolute magnitude of their log2 fold change**. The table includes the feature's `DataQuality` flag, the predicted compound name, and the underlying left and right group abundances (`leftAbundance`, `rightAbundance`) that were used to compute the fold change ratio.

**Note:** If a feature is present in the left group but absent from the right group, the fold change is set to `Double.POSITIVE_INFINITY` by SIRIUS. These entries are filtered out to retain only meaningful, finite fold changes.

**Note:** The table reflects only features in `aligned_features`, so the quality exclusion from Section 3.2 is automatically propagated here. It also includes only features that posses a structural annotation.

In [ ]:
import pandas as pd
import numpy as np

def get_annotated_fold_changes_df(statistics_table, group1, group2, aligned_features):
    """Return a DataFrame of log2 fold changes for all CSI:FingerID-annotated features,
    ordered by absolute log2 fold change (descending)."""
    col_idx, invert = None, False
    for idx, (left, right) in enumerate(zip(
        statistics_table.column_left_groups,
        statistics_table.column_right_groups
    )):
        if left == group1 and right == group2:
            col_idx = idx; break
        elif left == group2 and right == group1:
            col_idx = idx; invert = True; break

    if col_idx is None:
        return pd.DataFrame()

    feature_meta = {}
    for f in aligned_features:
        if f.top_annotations and f.top_annotations.structure_annotation is not None:
            name = f.top_annotations.structure_annotation.structure_name or ""
            feature_meta[f.aligned_feature_id] = (f.quality, name)

    rows = []
    for row_idx, feature_id in enumerate(statistics_table.row_ids):
        if feature_id not in feature_meta:
            continue
        raw = statistics_table.values[row_idx][col_idx]
        if raw <= 0:
            continue
        log2_fc = math.log2(raw) * (-1 if invert else 1)
        quality, compound_name = feature_meta[feature_id]
        left_abundance  = statistics_table.left_abundances[row_idx][col_idx]
        right_abundance = statistics_table.right_abundances[row_idx][col_idx]
        rows.append({
            'feature_id':     feature_id,
            'compound_name':  compound_name,
            'quality':        quality,
            'log2_fc':        log2_fc,
            'fold_change':    2 ** log2_fc,
            'direction':      f"{group1}/{group2}",
            'leftAbundance':  left_abundance,
            'rightAbundance': right_abundance,
        })

    return (
        pd.DataFrame(rows)
        .loc[lambda d: ~np.isinf(d['log2_fc'])]
        .assign(abs_log2_fc=lambda d: d['log2_fc'].abs())
        .sort_values('abs_log2_fc', ascending=False)
        .drop(columns='abs_log2_fc')
        .reset_index(drop=True)
    )


fc_df = get_annotated_fold_changes_df(feature_statistics_table, "Young Leaves", "Old Leaves", aligned_features)
print(f"{len(fc_df)} annotated features with computable fold change")
fc_df

The `quality` column allows immediate flagging of low-confidence hits. A `BAD` or `LOWEST` quality feature at the top of the ranking should be interpreted with caution, as its fold change magnitude may be inflated by sparse detection across replicates. This of course depends on what quality features we have filtered out before.

In [ ]:
top_id = fc_df.iloc[0]['feature_id']
top_feature = api.features().get_aligned_feature(
    project_info.project_id, top_id,
    opt_fields=[AlignedFeatureOptField.TOPANNOTATIONS]
)
print(f"Highest FC feature quality:     {top_feature.quality}")
print(f"Log2 fold change:               {fc_df.iloc[0]['log2_fc']}")
print(f"Structure annotation:           {top_feature.top_annotations.structure_annotation.structure_name}")
print(f"Left abundance  (Young Leaves):  {fc_df.iloc[0]['leftAbundance']}")
print(f"Right abundance (Old Leaves):    {fc_df.iloc[0]['rightAbundance']}")

The quality exclusion in Section 3.2 already removed the lowest-confidence features before computation. For exploratory purposes it is also possible to apply an additional post-hoc filter directly on `fc_df` without rerunning the job, for instance to focus temporarily on `DECENT` and `GOOD` features only. This changes the top-ranking feature accordingly.

In [ ]:
fc_df_high_quality = fc_df[fc_df['quality'].isin([DataQuality.GOOD])]
fc_df_high_quality

In [ ]:
top_id = fc_df_high_quality.iloc[0]['feature_id']
top_feature = api.features().get_aligned_feature(
    project_info.project_id, top_id,
    opt_fields=[AlignedFeatureOptField.TOPANNOTATIONS]
)
print(f"Highest FC feature quality:     {top_feature.quality}")
print(f"Log2 fold change:               {fc_df_high_quality.iloc[0]['log2_fc']}")
print(f"Structure annotation:           {top_feature.top_annotations.structure_annotation.structure_name}")
print(f"Left abundance  (Young Leaves):  {fc_df_high_quality.iloc[0]['leftAbundance']}")
print(f"Right abundance (Old Leaves):    {fc_df_high_quality.iloc[0]['rightAbundance']}")

## 6. Compound Class-Level Fold Change Analysis

Individual feature fold changes can be noisy. Aggregating at the level of compound classes provides a more robust and interpretable picture of which biosynthetic pathways or chemical families are differentially represented between sample groups. Doing this internally in SIRIUS carries an important advantage:

**Compound class fold changes are not derived by counting annotations over features, but are computed from the actual summed abundance of the annotated features within each class.**

Because the SIRIUS computation job in Section 3.3 was restricted to the quality-filtered feature set, the compound class statistics computed here are based exclusively on that same set.

SIRIUS supports two classification systems:
- **NPC** (Natural Products Classifier): three-level hierarchy of Pathway, Superclass, and Class.
- **ClassyFire**: hierarchical chemical ontology with up to nine levels.

Fold changes for both systems are computed below and visualised as bar charts, strip plots, and sunburst charts. Each compound class entry includes the aggregated left and right abundances (`leftAbundances`, `rightAbundances`) used to compute the fold change.

In [ ]:
compound_class_fold_change_submission = FoldChangeJobSubmission.from_dict({
    "leftRunGroup": "Young Leaves",
    "rightRunGroup": "Old Leaves",
    "aggregationTypes": [AggregationType.AVG],
    "quantificationMeasures": [QuantMeasure.AREA_UNDER_CURVE]
})

npc_fc_job = api.npc_class_statistics().compute_npc_class_fold_changes_experimental(
    project_info.project_id, compound_class_fold_change_submission
)
api.wait_for_job_completion(project_info.project_id, npc_fc_job.id)

classyfire_fc_job = api.classyfire_class_statistics().compute_classyfire_class_fold_changes_experimental(
    project_info.project_id, compound_class_fold_change_submission
)
api.wait_for_job_completion(project_info.project_id, classyfire_fc_job.id)

In [ ]:
npc_statistics_table = api.npc_class_statistics().get_npc_class_fold_change_table_experimental(
    project_info.project_id,
    aggregation=AggregationType.AVG,
    quantification=QuantMeasure.AREA_UNDER_CURVE
)

classyfire_statistics_table = api.classyfire_class_statistics().get_classyfire_class_fold_change_table_experimental(
    project_info.project_id,
    aggregation=AggregationType.AVG,
    quantification=QuantMeasure.AREA_UNDER_CURVE
)

### 6.1 Parse Class Statistics Tables

We convert the raw statistics tables into tidy DataFrames sorted by absolute log2 fold change. Each row retains the classification level, enrichment direction, log2 fold change, and the underlying left and right group abundances.

In [ ]:
def parse_class_statistics_table(statistics_table, group1, group2):
    """Convert a class-level statistics table to a tidy DataFrame."""
    col_idx, invert = None, False
    for idx, (left, right) in enumerate(zip(
        statistics_table.column_left_groups,
        statistics_table.column_right_groups
    )):
        if left == group1 and right == group2:
            col_idx = idx; break
        elif left == group2 and right == group1:
            col_idx = idx; invert = True; break

    assert col_idx is not None, f"Comparison {group1} vs {group2} not found."

    rows = []
    for row_idx, row_id in enumerate(statistics_table.row_ids):
        left_abundance  = statistics_table.left_abundances[row_idx][col_idx]
        right_abundance = statistics_table.right_abundances[row_idx][col_idx]
        raw = statistics_table.values[row_idx][col_idx]
        if raw <= 0:
            continue
        log2_fc = math.log2(raw) * (-1 if invert else 1)
        total_ab = (left_abundance or 0) + (right_abundance or 0)
        rows.append({
            "row_id":         row_id,
            "name":           statistics_table.row_names[row_idx],
            "level":          statistics_table.row_levels[row_idx],
            "log2_fc":        log2_fc,
            "abs_log2_fc":    abs(log2_fc),
            "enriched_in":    group1 if log2_fc > 0 else group2,
            "leftAbundance":  left_abundance,
            "rightAbundance": right_abundance,
            "totalAbundance": total_ab,
        })

    df = pd.DataFrame(rows).sort_values("abs_log2_fc", ascending=False)
    print(f"{len(df)} class entries retrieved.")
    return df


npc_df        = parse_class_statistics_table(npc_statistics_table,        "Young Leaves", "Old Leaves")
classyfire_df = parse_class_statistics_table(classyfire_statistics_table, "Young Leaves", "Old Leaves")

### 6.2 Do Rosmarinic Acid's Predicted Compound Classes All Follow the Same Old-Enriched Trend?

Rosmarinic acid is enriched in old leaves. We ask whether this trend holds for all compound classes that CANOPUS assigned to its base feature, or whether any class shows higher abundance in young leaves. A counter-trend class warrants a closer look at which feature is driving it.

Independent on hierarchy level, we select all classes with a predicted probability of at least 50%, then check its enrichment direction in `npc_df` and `classyfire_df`.

In [ ]:
canopus_result = api.features().get_canopus_prediction(
    project_info.project_id,
    rosmarinic_acid_base_feature.aligned_feature_id,
    rosmarinic_acid_base_feature.top_annotations.formula_annotation.formula_id
)

In [ ]:
ros_npc_classes = [c.name for c in canopus_result.npc_classes if c.probability >= 0.5]


print("Predicted NPC classes of rosmarinic acid base feature:", ros_npc_classes)

npc_lookup = npc_df.set_index('name')['enriched_in'].to_dict()

young_enriched_classes = []
for cls_name in ros_npc_classes:
    direction = npc_lookup.get(cls_name)
    if direction is None:
        print(f"  {cls_name}: not found in NPC fold change table (below threshold or absent)")
    elif direction == "Young Leaves":
        print(f"  {cls_name}: *** enriched in YOUNG Leaves (counter-trend!) ***")
        young_enriched_classes.append(cls_name)
    else:
        print(f"  {cls_name}: enriched in Old Leaves (consistent with rosmarinic acid)")

In [ ]:
ros_classyfire_classes = [c.name for c in canopus_result.classy_fire_classes if c.probability >= 0.5]

print("Predicted classyfire classes of rosmarinic acid base feature:", ros_classyfire_classes)

classyfire_lookup = classyfire_df.set_index('name')['enriched_in'].to_dict()

young_enriched_classes = []
for cls_name in ros_classyfire_classes:
    direction = classyfire_lookup.get(cls_name)
    if direction is None:
        print(f"  {cls_name}: not found in classyfire fold change table (below threshold or absent)")
    elif direction == "Young Leaves":
        print(f"  {cls_name}: *** enriched in YOUNG Leaves (counter-trend!) ***")
        young_enriched_classes.append(cls_name)
    else:
        print(f"  {cls_name}: enriched in Old Leaves (consistent with rosmarinic acid)")

We find that compound class fold changes seem to be in line with the rosmarinic acid fold change in all regards.

### 6.3 Visualisation Utilities

Three complementary plot types are provided for exploring class-level fold changes:

- **Bar chart**: best for inspecting a single hierarchy level with named classes.
- **Strip plot**: overview across all hierarchy levels, revealing at which levels the strongest enrichment signals occur.
- **Sunburst chart**: hierarchical view coloured by fold change and sized by total class abundance (sum of left and right abundances), providing an integrated picture of the entire classification tree.

All three functions accept a `threshold` argument (default: 1.0 log2 units, i.e. 2-fold) and an optional `levels` filter. Blue indicates enrichment in *Young Leaves*; orange indicates enrichment in *Old Leaves*. Adjust `ABS_LOG2_FC_THRESHOLD` to suit the dynamic range of your dataset.

In [ ]:
ABS_LOG2_FC_THRESHOLD = 1
COLOR_GROUP1 = "#378ADD"
COLOR_GROUP2 = "#D85A30"


def _filter(df, threshold, levels):
    mask = df["abs_log2_fc"] >= threshold
    if levels is not None:
        mask &= df["level"].isin(levels)
    return df[mask]


def _title(group1, group2, levels, threshold):
    lvl_str = ", ".join(levels) if levels else "all levels"
    return f"{group1} vs {group2} | {lvl_str} | |log2 FC| >= {threshold}"


def plot_bar(df, group1, group2, threshold=ABS_LOG2_FC_THRESHOLD, levels=None):
    sub = _filter(df, threshold, levels)
    colors = [COLOR_GROUP1 if v > 0 else COLOR_GROUP2 for v in sub["log2_fc"]]
    fig = go.Figure(go.Bar(
        x=sub["log2_fc"], y=sub["name"],
        orientation="h",
        marker_color=colors,
        hovertemplate="%{y}<br>log2 FC: %{x:.2f}<extra></extra>"
    ))
    fig.update_layout(
        title=_title(group1, group2, levels, threshold),
        xaxis_title="log2 FC",
        xaxis=dict(zeroline=True, zerolinecolor="black", zerolinewidth=1),
        height=max(400, len(sub) * 20),
        template="plotly_white"
    )
    fig.show()


def plot_strip(df, group1, group2, threshold=ABS_LOG2_FC_THRESHOLD, levels=None):
    sub = _filter(df, threshold, levels)
    fig = px.strip(
        sub, x="log2_fc", y="level",
        color="enriched_in",
        hover_name="name",
        color_discrete_map={group1: COLOR_GROUP1, group2: COLOR_GROUP2},
        category_orders={"level": sorted(sub["level"].unique())},
        title=_title(group1, group2, levels, threshold)
    )
    fig.update_layout(
        xaxis_title="log2 FC",
        xaxis=dict(zeroline=True, zerolinecolor="black", zerolinewidth=1),
        template="plotly_white"
    )
    fig.show()


def plot_abundance_sunbursts(df, group1, group2, levels=None):
    """Side-by-side sunbursts of raw group abundances per class.

    Slice size encodes the group abundance of the class.
    These plots give direct context for the fold change sunburst that follows.
    """
    sub = df.copy()
    if levels is not None:
        sub = sub[sub["level"].isin(levels)]
    sub = sub.dropna(subset=["leftAbundance", "rightAbundance"])

    fig = go.Figure()
    fig.add_trace(go.Sunburst(
        labels=sub["name"],
        parents=sub["level"],
        values=sub["leftAbundance"],
        name=group1,
        domain={"x": [0, 0.48]},
        marker_colorscale="Blues",
        hovertemplate="<b>%{label}</b><br>Abundance: %{value:.2e}<extra></extra>",
        branchvalues="total",
    ))
    fig.add_trace(go.Sunburst(
        labels=sub["name"],
        parents=sub["level"],
        values=sub["rightAbundance"],
        name=group2,
        domain={"x": [0.52, 1.0]},
        marker_colorscale="Oranges",
        hovertemplate="<b>%{label}</b><br>Abundance: %{value:.2e}<extra></extra>",
        branchvalues="total",
    ))
    lvl_str = ", ".join(levels) if levels else "all levels"

    fig.update_layout(
        title=f"Class Abundance: {group1} (left) vs {group2} (right) | {lvl_str}",
        annotations=[
            dict(text=group1, x=0.20, y=1.10, font_size=13, showarrow=False, xref="paper", yref="paper"),
            dict(text=group2, x=0.80, y=1.10, font_size=13, showarrow=False, xref="paper", yref="paper"),
        ],
        height=600,
        template="plotly_white",
        margin=dict(t=100, b=40, l=40, r=40)
    )
    fig.show()


def plot_sunburst(df, group1, group2, threshold=ABS_LOG2_FC_THRESHOLD, levels=None):
    """Sunburst where color encodes log2 fold change and slice size encodes total class abundance."""
    sub = _filter(df, threshold, levels)
    fig = px.sunburst(
        sub, path=["level", "name"],
        values="totalAbundance",
        color="log2_fc",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        hover_data={"log2_fc": ":.2f", "enriched_in": True, "leftAbundance": True, "rightAbundance": True},
        title=_title(group1, group2, levels, threshold)
    )
    fig.update_coloraxes(colorbar_title="log2 FC")
    fig.update_layout(
        template="plotly_white"
    )
    fig.show()

#### NPC: Bar Charts by Hierarchy Level

In [ ]:
for level in npc_df["level"].unique():
    plot_bar(npc_df, "Young Leaves", "Old Leaves", levels=[level])

#### NPC: Strip Plot (All Levels)

In [ ]:
plot_strip(npc_df, "Young Leaves", "Old Leaves")

#### NPC: Raw Group Abundance Sunbursts

Before examining the fold change sunburst, we first inspect the raw group abundances per compound class side by side. It is important to know that here, ratios between both groups do not apply. A slice of one class might simply be bigger due to the total abundance of all classes being much lower than of the other class. Hovering over the slices, we can see the actual abundances, which are the basis for the fold change analysis.

In [ ]:
for level in npc_df["level"].unique():
    plot_abundance_sunbursts(npc_df, "Young Leaves", "Old Leaves", levels=[level])

#### NPC: Fold Change Sunburst (Hierarchical Overview)

The fold change sunburst encodes two quantities simultaneously: **color** reflects the log2 fold change (blue = enriched in Young Leaves, red = enriched in Old Leaves), and **slice size** encodes the total abundance of each class (sum of left and right group abundances). Classes with large slices contribute the most to the overall signal; the color then reveals whether that abundant class is preferentially found in one group.

In [ ]:
for level in npc_df["level"].unique():
    plot_sunburst(npc_df, "Young Leaves", "Old Leaves", levels=[level])

#### ClassyFire: Bar Charts by Hierarchy Level

Strip and sunburst plots are omitted for ClassyFire because its up to nine hierarchical levels make a flat sunburst layout less interpretable. Adapt as needed for your dataset.

In [ ]:
for level in classyfire_df["level"].unique():
    plot_bar(classyfire_df, "Young Leaves", "Old Leaves", levels=[level])

## 6.4 Further Analysis

The plots above illustrate one possible approach, but the most informative statistics and visualisations will depend on the specific dataset and research question. We encourage the reader to extend the analysis below using the tidy DataFrames (`npc_df`, `classyfire_df`, `fc_df`) and the raw statistics tables as starting points.

After completing the analysis, proceed to Section 6 for cleanup.

## 7. Cleanup

Close the project space to release file handles and shut down the SIRIUS REST service. If SIRIUS should remain running for further analysis, omit `sdk.shutdown_sirius()`.

All data, annotations, and fold change tables remain saved in the project space file and can be reopened at any time via `api.projects().open_project()`.

In [ ]:
api.projects().close_project(project_info.project_id)
sdk.shutdown_sirius()